# RFD4-Proteina Generative Protein & Binder Design with `foldrun-shim`

This notebook adapts the official [**`nvidia/RFD4-Proteina`**](https://huggingface.co/nvidia/RFD4-Proteina) ([`rfd4-proteina.org`](https://rfd4-proteina.org) / [**`NVIDIA-Digital-Bio/proteina`**](https://github.com/NVIDIA-Digital-Bio/proteina) + `atomworks` / `rfproteina`) workflows across all three generative design modes—**Target-Conditioned Binder Design (`binder_design`)**, **Motif Scaffolding (`motif_scaffolding`)**, and **Unconditional Monomer Generation (`unconditional_monomer`)**—to execute on **FoldRun**'s auto-scaling Google Cloud Vertex AI GPU Pipelines via **`foldrun-shim`**.


---

## Original Notebook Reference, Credits & Citations

- **Original Public Examples & Documentation**:
  - Official Model Card & Inference Examples: [`nvidia/RFD4-Proteina` on HuggingFace](https://huggingface.co/nvidia/RFD4-Proteina) and [`rfd4-proteina.org`](https://rfd4-proteina.org)
  - Official Proteina Codebase & Notebooks: [`NVIDIA-Digital-Bio/proteina`](https://github.com/NVIDIA-Digital-Bio/proteina) (`atomworks` / `rfproteina`)
- **Authors & Credits**: Tomas Geffner, Kieran Didi, Zuobai Zhang, Danny Reidenbach, Zhonglin Cao, Jason Yim, Mario Geiger, Christian Dallago, Emine Kucukbenli, Karsten Kreis, Arash Vahdat, and collaborators (**NVIDIA Digital Biology Research & Institute for Protein Design (IPD), University of Washington**).
- **License**: NVIDIA Open Model License (`nvidia/RFD4-Proteina`) / Apache License 2.0 (`NVIDIA-Digital-Bio/proteina`).
- **Primary Citations**:
  1. Geffner, T., Didi, K., Zhang, Z. *et al.* "Proteina: Scaling Flow-Based Protein Structure Generative Models." *International Conference on Learning Representations (ICLR)* (2025). [arXiv:2503.00710](https://arxiv.org/abs/2503.00710)
  2. NVIDIA Digital Biology & IPD. "RFD4-Proteina: Joint Sequence-Structure Flow-Matching for De Novo Protein and Binder Design." (2025). [huggingface.co/nvidia/RFD4-Proteina](https://huggingface.co/nvidia/RFD4-Proteina)

---

## Why Run RFD4-Proteina Through `foldrun-shim` Instead of a Standard Colab / Local Environment?

| Capability | Public RFD4-Proteina Colab / Local (`!rfd4-generate ...`) | FoldRun Cloud Execution via `foldrun-shim` |
| :--- | :--- | :--- |
| **Model & Autoencoder Weights (`nvidia/RFD4-Proteina`)** | Downloads ~1.7 GB flow-matching + latent autoencoder weights from HuggingFace on every session reset; `hf_xet` transfers frequently fail over network filesystems with `CasObjectError: PresignedUrlExpirationError`. | Pre-staged once via Cloud Batch (`HF_HUB_DISABLE_XET=1`) onto a 2.5 TB `BASIC_SSD` Cloud Filestore NFS share at `/mnt/nfs/foldrun/rfd4_proteina/cache/hub` and mounted directly into every GPU worker (`HF_HOME`, `HF_HUB_CACHE`, `RFD4_CACHE`, `DATA_PATH`). |
| **Target Residue Indexing (`auth_seq_id` vs. `label_seq_id`)** | `atomworks` re-indexes parsed target chains into 1-indexed contiguous `label_seq_id` coordinates (`1..N`). Passing crystallographic `auth_seq_id` spans (e.g., `A18-132` when residues 1–17 are unresolved in PD-L1 `4Z18`) crashes with `IndexError: index 114 is out of bounds for axis 0 with size 114`. | FoldRun's `gemmi`-powered resolver automatically inspects `_atom_site.auth_seq_id` and `_atom_site.label_seq_id`, computing the 1-indexed `label_seq_id` span (`A1-115`) and translating crystallographic hotspot numbers (`A54, A56, A66` $\rightarrow$ `37, 39, 49`) before launching `rfd4-generate`. |
| **Hotspot Condition Token Validation (`C_HOT`)** | `rfproteina` strictly validates condition keys against `{C_ADD, C_HOT, C_TERM, H, E, L}`; using `HOTSPOT` raises `AssertionError: Invalid conditions: {'HOTSPOT'}`. | Automatically translates high-level `binding_residues` into `C_HOT: {True: [{'select': "sel('A/*/38/[CA,CB]')"}]}` alongside `C_CRDGRP` and `C_SEQ`. |
| **Generated `.cif` `NaN` B-Factors & `.cif.gz` Outputs** | `rfd4-generate` writes `NaN` into `_atom_site.B_iso_or_equiv` and may emit compressed `.cif.gz` / `.pdb.gz` files, causing RFC 8259 JSON parse errors (`Unexpected token 'N'`) and broken `3Dmol.js` pLDDT coloring. | `predict_rfd4_proteina` converts `.pdb` / `.cif.gz` / `.pdb.gz` into uncompressed `.cif` via `gemmi`, and FoldRun sanitizes `NaN` / `Infinity` B-factors for valid JSON and `py3Dmol` / `3Dmol.js` rendering. |
| **CUDA Custom Kernels & `cuequivariance` Stability** | Custom C++/CUDA kernel compilation or host driver version mismatches crash `rfd4-generate` at startup on heterogeneous cloud GPUs. | Proactively exports `PROTEINA_DISABLE_CUSTOM_KERNELS=1` and `CUEQUIVARIANCE_OPS_FALLBACK=1` inside `predict_rfd4_proteina` to guarantee reliable PyTorch execution. |
| **Multi-Seed GPU Fan-Out (`A100` / `A100_80GB` / `L4`)** | Runs replicates sequentially on a single memory-constrained Colab GPU. | Two-stage Vertex AI Pipeline (`ConfigureSeeds -> ParallelFor[PredictOp]`) fans out `--num_model_seeds` (`1..50`) in parallel across **A100 40GB** ($\le 1500$ tokens), **A100 80GB** ($> 1500$ tokens), or cost-optimized **L4** GPUs (`gpu_type="L4"`) with DWS `FLEX_START`. |
| **Disconnect Resilience (`.foldrun_job.json`)** | Closing the browser tab or losing Wi-Fi terminates the generation job and loses all sampled structures. | Persists `<out_dir>/.foldrun_job.json` + server-side `input_sha256` labels so re-running an interrupted cell re-attaches to the running Vertex AI `PipelineJob` and downloads `<stem>/ranked_designs/` and `<stem>/predictions/`. |


## 1. Prerequisites, Authentication & Installing `foldrun-shim` from GCS

During FoldRun deployment (`./deploy-all.sh` or `./deploy-all.sh --steps build --build-target sdk`), Cloud Build packages `foldrun-shim` into a pure-Python wheel and uploads both the wheel and all 7 modified public notebooks to your project's FoldRun GCS bucket under `gs://<PROJECT_ID>-foldrun-data/sdk/`:
- **Pre-Built Wheel**: `gs://<PROJECT_ID>-foldrun-data/sdk/latest_build` (and dated incremental builds)
- **Modified Notebooks**: `gs://<PROJECT_ID>-foldrun-data/sdk/notebooks/<model>/*.ipynb`

### Step-by-Step Setup (Executed in the Next Cell)

1. **Step 1 — Authenticate to Google Cloud First**:
   - **Google Colab**: Calls `from google.colab import auth; auth.authenticate_user()` so `gcloud storage cp` and `gcloud auth print-identity-token` have valid credentials before downloading the wheel.
   - **Colab Enterprise / Vertex AI Workbench**: Authenticated automatically via the runtime's attached service account and the Google Cloud Metadata Server.
   - **Local Jupyter Kernel**: Authenticate beforehand with `gcloud auth login`.
2. **Step 2 — Automatic Configuration Loading (`.env`)**:
   - FoldRun's deployment script (`./deploy-all.sh`) generates local `.env` files in `sdk/foldrun-shim/notebooks/` and uploads `gs://${FOLDRUN_GCS_BUCKET}/sdk/.env`.
   - The cell below automatically loads `.env` (searching `./.env`, `../.env`, `/content/.env`, or downloading from `gs://${FOLDRUN_GCS_BUCKET}/sdk/.env`), populating `GCP_PROJECT_ID`, `GCP_REGION`, `FOLDRUN_GCS_BUCKET`, `FOLDRUN_GATEWAY_URL`, and `FOLDRUN_VIEWER_URL`.
3. **Step 3 — Download & Install `foldrun-shim` from the FoldRun GCS Bucket**:
   Requires `roles/storage.objectViewer` on `gs://<PROJECT_ID>-foldrun-data` (ideally scoped to the `sdk/` prefix via an IAM condition):
   ```bash
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/latest_build /tmp/
   WHL=$(cat /tmp/latest_build)
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/${WHL} /tmp/
   pip install -q /tmp/${WHL}
   ```
4. **Step 4 — Configure `FOLDRUN_GATEWAY_URL` & OIDC Gateway Auth (`roles/run.invoker`)**:
   `foldrun_shim.auth.fetch_gateway_headers()` automatically attaches `Authorization: Bearer <OIDC_ID_TOKEN>` to every gateway request via:
   1. `FOLDRUN_ID_TOKEN` (if explicitly set),
   2. `FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT` (`gcloud auth print-identity-token --impersonate-service-account=...`),
   3. Google Cloud Metadata Server (`google.oauth2.id_token.fetch_id_token` on Vertex AI Workbench / Colab Enterprise),
   4. `gcloud auth print-identity-token` fallback (Colab / local session).

In [ ]:
# @title 1. Authenticate to Google Cloud & Install `foldrun-shim` from GCS
import csv
import json
import math
import os
import re
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path


def _ensure_scripts_on_path() -> None:
    """Ensure pip console scripts (including --user installs) are discoverable on PATH."""
    script_dirs: list[str] = []
    for scheme in (None, f"{os.name}_user"):
        try:
            p = (
                sysconfig.get_path("scripts")
                if scheme is None
                else sysconfig.get_path("scripts", scheme)
            )
            if p:
                script_dirs.append(p)
        except Exception:
            pass
    script_dirs.append(str(Path(sys.executable).parent))
    current_parts = os.environ.get("PATH", "").split(os.pathsep) if os.environ.get("PATH") else []
    for d in reversed(script_dirs):
        if d and d not in current_parts:
            current_parts.insert(0, d)
    os.environ["PATH"] = os.pathsep.join(current_parts)


def _is_placeholder_value(val: str) -> bool:
    lowered = (val or "").strip().lower()
    return not lowered or any(
        tok in lowered
        for tok in (
            "your-project-foldrun-data",
            "your-project",
            "your_project",
            "example",
            "<",
        )
    )


def _is_placeholder_bucket(bucket: str) -> bool:
    clean = (bucket or "").strip().removeprefix("gs://").rstrip("/")
    return _is_placeholder_value(clean)


_ensure_scripts_on_path()


def _load_foldrun_dotenv(custom_path: Path | None = None) -> Path | None:
    """Load key=value pairs from a local .env file generated by deploy-all.sh."""
    if custom_path is not None:
        candidates = [custom_path]
    else:
        candidates = []
        if os.environ.get("FOLDRUN_ENV_FILE"):
            candidates.append(Path(os.environ["FOLDRUN_ENV_FILE"]))
        candidates.extend(
            [Path(".env"), Path("../.env"), Path("../../.env"), Path("/content/.env")]
        )
    loaded_from: Path | None = None
    for path in candidates:
        if not path.is_file():
            continue
        for raw_line in path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k = re.sub(r"^export(?=\s)\s*", "", k.strip())
            v = v.strip().strip("'").strip('"')
            if k and v:
                existing = os.environ.get(k, "")
                if not existing or _is_placeholder_value(existing):
                    os.environ[k] = v
        if loaded_from is None:
            loaded_from = path.resolve()
        if custom_path is not None or not _is_placeholder_value(
            os.environ.get("FOLDRUN_GATEWAY_URL", "")
        ):
            return path.resolve()
    return loaded_from


_dotenv_path = _load_foldrun_dotenv()
if _dotenv_path:
    print(f"Loaded FoldRun configuration from {_dotenv_path}")

os.environ.setdefault("FOLDRUN_GCS_BUCKET", "your-project-foldrun-data")
os.environ.setdefault("FOLDRUN_GATEWAY_URL", "https://foldrun-gateway-example-uc.a.run.app")
# Optional: set FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT if delegating to an invoker service account
# os.environ["FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT"] = "foldrun-invoker@your-project.iam.gserviceaccount.com"

# 1. Authenticate to Google Cloud first (required in standard Google Colab before accessing GCS)
if "google.colab" in sys.modules:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()


def _install_foldrun_shim_from_gcs(gcs_bucket: str) -> None:
    """Download latest foldrun_shim wheel via latest_build from gs://<bucket>/sdk/ and install via pip."""
    clean_bucket = (gcs_bucket or "").strip().removeprefix("gs://").rstrip("/")
    bucket_configured = not _is_placeholder_bucket(clean_bucket)

    if bucket_configured and _is_placeholder_value(os.environ.get("FOLDRUN_GATEWAY_URL", "")):
        try:
            env_dst = Path("/tmp/foldrun_sdk.env")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/.env",
                    str(env_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and env_dst.is_file():
                if _load_foldrun_dotenv(env_dst):
                    print(f"Loaded remote FoldRun configuration from gs://{clean_bucket}/sdk/.env")
        except Exception as exc:
            print(f"Notice: could not load gs://{clean_bucket}/sdk/.env: {exc}")

    whl_filename = "foldrun_shim-0.1.0-py3-none-any.whl"
    if bucket_configured:
        try:
            latest_build_dst = Path("/tmp/foldrun_latest_build")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/latest_build",
                    str(latest_build_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and latest_build_dst.is_file():
                candidate = latest_build_dst.read_text(encoding="utf-8").strip()
                if candidate.endswith(".whl"):
                    whl_filename = candidate
                    print(
                        f"Detected latest FoldRun SDK build from gs://{clean_bucket}/sdk/latest_build: {whl_filename}"
                    )
        except Exception as exc:
            print(f"Notice: could not read gs://{clean_bucket}/sdk/latest_build: {exc}")

    whl_path = Path(f"/tmp/{whl_filename}")
    marker_path = Path(f"/tmp/{whl_filename}.installed")
    if marker_path.exists():
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            return
        except ImportError:
            pass

    if not bucket_configured:
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            print(
                "Notice: FOLDRUN_GCS_BUCKET is unconfigured; using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"foldrun_shim is not installed. Set FOLDRUN_GCS_BUCKET='<PROJECT_ID>-foldrun-data' "
                f"so this cell can download gs://<PROJECT_ID>-foldrun-data/sdk/{whl_filename}."
            )

    gcs_uri = f"gs://{clean_bucket}/sdk/{whl_filename}"
    print(f"Downloading {whl_filename} from {gcs_uri}...")
    cp_res = subprocess.run(
        ["gcloud", "storage", "cp", gcs_uri, str(whl_path)],
        check=False,
        capture_output=True,
        text=True,
    )
    if cp_res.returncode != 0:
        err_line = (
            (cp_res.stderr or cp_res.stdout or "gcloud storage cp failed").strip().splitlines()[-1]
        )
        try:
            import foldrun_shim  # noqa: F401

            _ensure_scripts_on_path()
            print(
                f"Warning: could not download {gcs_uri} ({err_line}); using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"Failed to download {gcs_uri} ({err_line}) and foldrun_shim is not installed locally."
            )

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", str(whl_path)],
        check=True,
    )
    _ensure_scripts_on_path()
    marker_path.write_text("installed", encoding="utf-8")
    print(f"Installed {whl_filename} successfully.")


_install_foldrun_shim_from_gcs(os.environ["FOLDRUN_GCS_BUCKET"])
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "py3Dmol",
        "matplotlib",
        "numpy",
        "pyyaml",
    ],
    check=False,
)

import yaml

import foldrun_shim
from foldrun_shim import rfd4_proteina
from foldrun_shim.bundler import bundle_local_inputs

print(f"foldrun-shim version : {getattr(foldrun_shim, '__version__', 'unknown')}")
print(f"rfd4-generate in PATH: {shutil.which('rfd4-generate') or 'available via foldrun_shim.cli'}")
print(f"FOLDRUN_GATEWAY_URL  : {os.environ['FOLDRUN_GATEWAY_URL']}")

In [ ]:
# @title 1B. (Optional) Check Live GPU Availability & Configure On-Demand vs. DWS Scheduling
# @markdown Thin-permission notebook users (who only have `roles/run.invoker` on the FoldRun Gateway)
# @markdown can inspect live GPU availability, active/queued jobs, and regional quotas directly from Python
# @markdown without needing GCP project-level GCE or ServiceUsage IAM permissions.
# @markdown
# @markdown **CLI Equivalents:**
# @markdown - `!foldrun gpus` (inspect live GPU availability and queue depths)
# @markdown - `!foldrun status <job_id>` (inspect status of an active or queued job)
# @markdown - `!foldrun cancel <job_id>` (cancel a queued or running job)
# @markdown - GCP Project Admins only: `!gcloud alpha services quota list --service=aiplatform.googleapis.com --consumer=projects/$GCP_PROJECT_ID`

import os

try:
    from IPython.display import display
except ImportError:
    display = print

# 1. Query live GPU availability and queue status via FoldRun Gateway
try:
    import foldrun_shim

    gpu_status = foldrun_shim.check_gpus()
    display(gpu_status)
except Exception as exc:
    print(f"Notice: Could not query live GPU status via FoldRun Gateway: {exc}")
    print(
        "You can still proceed with prediction using default scheduling or environment variables below."
    )

# 2. Scheduling & Hardware Configuration (Optional Overrides)
# Uncomment or adjust any of the following environment variables to control hardware & scheduling:

# GPU Architecture: "auto" (default), "L4" (fast & cost-effective), "A100" (40GB), or "A100_80GB"
# os.environ["FOLDRUN_GPU_TYPE"] = "L4"

# Scheduling Strategy:
# - "true"  -> Dynamic Workload Scheduler (DWS) FLEX_START (saves 45-68% cost; may queue if regional DWS capacity is constrained)
# - "false" -> Immediate On-Demand STANDARD GPUs (provisions immediately without DWS queueing)
# os.environ["FOLDRUN_ENABLE_FLEX_START"] = "false"

# Optional Queue Timeout: Automatically cancel the job if queued in DWS longer than N seconds
# os.environ["FOLDRUN_QUEUE_TIMEOUT_SEC"] = "300"

---
## Part A — Original Upstream Example (Unmodified / Verbatim Workflow)

### Original HuggingFace `nvidia/RFD4-Proteina` Quickstart Examples

Below is the exact quickstart command from the official [`nvidia/RFD4-Proteina` HuggingFace Model Card](https://huggingface.co/nvidia/RFD4-Proteina):
```bash
rfd4-generate inputs=data/benchmarks/monomer/monomer-short.cif \
  hf_ckpt_path=nvidia/RFD4-Proteina \
  hf_ckpt_revision=a06e2509c7033b010fc43614e9fd840303588502 \
  num_replicates=1 out_dir=./runs/first '~metrics' compile_lengths=null
```

> **How `foldrun-shim` executes this without local installation**:
> In standard upstream environments, running this command requires a local clone of `NVIDIA-Digital-Bio/proteina` or importing the package (`inputs=$(python -c "import rfproteina, pathlib; print(pathlib.Path(rfproteina.__file__).parent / 'data/benchmarks/monomer/monomer-short.cif')")`).
> With `foldrun-shim`, **`rfproteina` does not even need to be installed locally**! `foldrun-shim` recognizes `data/benchmarks/...` paths, transparently packages them, and resolves the benchmark files directly inside `/opt/rfd4-proteina` on the Vertex AI GPU worker while stripping local checkpoint flags (`hf_ckpt_path`, `hf_ckpt_revision`).


In [ ]:
# Exact upstream HuggingFace nvidia/RFD4-Proteina command verbatim:
!rfd4-generate inputs=data/benchmarks/monomer/monomer-short.cif hf_ckpt_path=nvidia/RFD4-Proteina hf_ckpt_revision=a06e2509c7033b010fc43614e9fd840303588502 num_replicates=1 out_dir=./runs/first '~metrics' compile_lengths=null

In [ ]:
# Upstream HuggingFace packaged Python snippet & benchmarks:
# In upstream: inputs=$(python -c "import rfproteina, pathlib; print(pathlib.Path(rfproteina.__file__).parent / 'data/benchmarks/monomer/monomer-short.cif')")
# With foldrun-shim, rfproteina is not required locally; benchmarks resolve inside /opt/rfd4-proteina on the Vertex AI worker.

# Upstream PPI Binder Benchmark (PD-L1):
!rfd4-generate inputs=data/benchmarks/ppi/pdl1_short.cif hf_ckpt_path=nvidia/RFD4-Proteina hf_ckpt_revision=a06e2509c7033b010fc43614e9fd840303588502 num_replicates=1 out_dir=./runs/pdl1 '~metrics' compile_lengths=null

# Upstream Motif Scaffolding Benchmark:
!rfd4-generate inputs=data/benchmarks/motif_scaffolding/02_1YCR.cif hf_ckpt_path=nvidia/RFD4-Proteina hf_ckpt_revision=a06e2509c7033b010fc43614e9fd840303588502 num_replicates=1 out_dir=./runs/motif '~metrics' compile_lengths=null

---
## Part B — FoldRun Additions (`foldrun-shim` Python API, `/mnt/nfs/foldrun` Shared Databases & Weights, Local Asset Bundling, and Multi-GPU Seed Parallelism)

FoldRun enhances RFD4-Proteina with production cloud capabilities:
1. **Pre-staged NFS weights (`/mnt/nfs/foldrun/rfd4_proteina/cache/hub`)**: Zero-download container startup bypassing HuggingFace Xet CAS network timeout errors (`HF_HUB_DISABLE_XET=1`).
2. **Automatic `auth_seq_id -> label_seq_id` and `C_HOT` resolution**: `gemmi` resolves crystal structure offsets to prevent `atomworks` `IndexError`.
3. **Local asset bundling**: Automatically packages custom local `.cif` files and YAML configs into `assets.tar.gz`.
4. **Multi-seed GPU fan-out**: Fans out parallel runs via `--num_model_seeds=5` across Vertex AI A100/L4 GPUs.
5. **Python Helper API (`rfd4_proteina.predict`)**: Clean programmatic interface with disconnect recovery (`.foldrun_job.json`).


## 2. How the FoldRun Database & RFD4-Proteina Runtime Safeguards Work

### 2.1 Pre-Staged NFS Model + Autoencoder Cache (`/mnt/nfs/foldrun/rfd4_proteina/cache`)
In `foldrun-agent/foldrun_app/databases.yaml`, FoldRun provisions the `rfd4_proteina_cache` entry onto the shared **2.5 TB `BASIC_SSD` Cloud Filestore NFS share** (`/mnt/nfs/foldrun/rfd4_proteina/cache/hub`) via Cloud Batch:
- **Why `HF_HUB_DISABLE_XET=1` is required**: HuggingFace's default `hf_xet` Content-Addressed Storage (CAS) downloader frequently fails with `CasObjectError: PresignedUrlExpirationError` when writing multi-file model snapshots across NFS mounts. Setting `HF_HUB_DISABLE_XET=1` forces standard HTTPS chunked downloads during Cloud Batch provisioning.
- **Zero-Download Container Startup**: Every `predict_rfd4_proteina` GPU task mounts `/mnt/nfs/foldrun` and exports:
  ```bash
  export HF_HOME=/mnt/nfs/foldrun/rfd4_proteina/cache
  export HF_HUB_CACHE=/mnt/nfs/foldrun/rfd4_proteina/cache/hub
  export RFD4_CACHE=/mnt/nfs/foldrun/rfd4_proteina/cache
  export DATA_PATH=/mnt/nfs/foldrun/rfd4_proteina/cache
  export PROTEINA_DISABLE_CUSTOM_KERNELS=1
  export CUEQUIVARIANCE_OPS_FALLBACK=1
  ```

### 2.2 Automatic `auth_seq_id -> label_seq_id` Mapping & `C_HOT` Translation
Upstream `rfd4-generate` (`atomworks` + `rfproteina`) enforces two strict input invariants:
1. **1-Indexed Contiguous `label_seq_id` Coordinates**: `atomworks` re-indexes each parsed target chain to `1..N`. In crystal structures where N-terminal residues are unresolved—such as human PD-L1 (`4Z18`), where Chain A starts at crystallographic residue `auth_seq_id = 18` and runs through `132` ($115$ resolved residues)—passing `contig: "80-120, /0, A18-132"` or hotspot `A54` directly causes `atomworks` to slice beyond index `114` and crash with:
   `IndexError: index 114 is out of bounds for axis 0 with size 114`.
2. **Strict Condition Token Allowlist (`{C_ADD, C_HOT, C_TERM, H, E, L}`)**: Passing `HOTSPOT` in `conditions:` raises:
   `AssertionError: Invalid conditions: {'HOTSPOT'}`.

When you submit a target `.cif` or 4-character PDB ID (`4Z18`), FoldRun's `predict_rfd4_proteina` component inspects `_atom_site.auth_asym_id`, `_atom_site.auth_seq_id`, and `_atom_site.label_seq_id` via `gemmi` to:
- Replace the target contig span with the exact 1-indexed `label_seq_id` range (`A1-115`), and
- Map crystallographic hotspot numbers (`A54`, `A56`, `A66`) to their 1-indexed `label_seq_id` counterparts (`37`, `39`, `49`) inside `C_HOT: {True: [{"select": "sel('A/*/37/[CA,CB]')"}, ...]}`.

### 2.3 Automatic `NaN` B-Factor Sanitization & `.cif.gz` Normalization
- `rfd4-generate` writes `NaN` in the `_atom_site.B_iso_or_equiv` column of generated structures and can emit gzip-compressed `.cif.gz` or `.pdb.gz` files.
- `predict_rfd4_proteina` normalizes all generated structures into uncompressed `.cif` files via `gemmi` (`design_seed<seed>_<i>.cif`), and FoldRun's analysis and viewer pipeline sanitizes `NaN`/`Infinity` values so RFC 8259 JSON serialization and `py3Dmol` / `3Dmol.js` rendering never fail.

In [ ]:
# Demonstrate FoldRun's two RFD4-Proteina data safeguards:
# 1. Crystallographic auth_seq_id (18..132) -> 1-indexed atomworks label_seq_id (1..115) & C_HOT translation
# 2. Sanitization of NaN values in _atom_site.B_iso_or_equiv from rfd4-generate .cif outputs

# Simulate PD-L1 (PDB: 4Z18 Chain A) where crystallographic residues 1-17 are unresolved:
# auth_seq_id runs 18..132, while mmCIF label_seq_id runs 1..115
auth_to_label_4z18 = {str(auth_id): str(auth_id - 17) for auth_id in range(18, 133)}
user_hotspots = ["A54", "A56", "A66"]  # PD-L1 interface hotspots (Ile54, Tyr56, Gln66)

hotspot_selects = []
for res_item in user_hotspots:
    ch_id, auth_num = res_item[0], res_item[1:]
    mapped_label_num = auth_to_label_4z18[auth_num]
    hotspot_selects.append({"select": f"sel('{ch_id}/*/{mapped_label_num}/[CA,CB]')"})

translated_rfd4_yaml = {
    "binder_design": {
        "input": "4z18.cif",
        "contig": f"80-120, /0, A1-{len(auth_to_label_4z18)}",
        "conditions": {
            "C_CRDGRP": {1: [{"select": "sel('A/*/*/[N,CA,C,O]')"}]},
            "C_SEQ": {True: [{"select": "sel('A')"}]},
            "C_HOT": {True: hotspot_selects},
        },
    }
}
print("=== 1. FoldRun auth_seq_id -> label_seq_id & C_HOT Translation (4Z18 Chain A) ===")
print(yaml.safe_dump(translated_rfd4_yaml, sort_keys=False))

# Demonstrate NaN B-factor sanitization on raw rfd4-generate mmCIF lines (column 15 / index 14 = B_iso_or_equiv)
raw_rfd4_cif_line = "ATOM 1 N N . MET B 1 1 ? 12.340 24.560 8.910 1.00 NaN 1 MET B N 1"
tokens = raw_rfd4_cif_line.split()
raw_b_factor = float(tokens[14])
sanitized_b_factor = 0.0 if (math.isnan(raw_b_factor) or math.isinf(raw_b_factor)) else raw_b_factor
tokens[14] = f"{sanitized_b_factor:.2f}"
print("=== 2. FoldRun NaN B-Factor Sanitization ===")
print(f"Raw rfd4-generate _atom_site line : {raw_rfd4_cif_line}")
print(f"Sanitized mmCIF _atom_site line   : {' '.join(tokens)}")
print(
    f"Strict RFC 8259 JSON check        : {json.dumps({'b_factor': sanitized_b_factor}, allow_nan=False)}"
)

---
## 3. Mode 1 — Target-Conditioned Binder Design (`binder_design`)

### Interface 1: Drop-In Native Hydra CLI (`!rfd4-generate ...`)

RFD4-Proteina supports two ways to specify a **Target-Conditioned Binder Design** job:
1. **Native `rfd4-generate` YAML (`pdl1_binder_design.yaml`)**: Adapted from the official [`rfd4-proteina.org`](https://rfd4-proteina.org) PD-L1 binder example, specifying:
   - `input`: Local `.cif`/`.pdb` path (automatically bundled into `assets.tar.gz` by `foldrun-shim`), `gs://` URI, or 4-character RCSB PDB ID.
   - `contig`: `"80-120, /0, A1-115"` (generates an 80–120 residue binder chain followed by chain break `/0` and target chain `A` residues `1-115`).
   - `conditions`: `C_CRDGRP` (rigid target backbone coordinates `[N,CA,C,O]`), `C_SEQ` (fixed target sequence `sel('A')`), and `C_HOT` (target interface hotspot `[CA,CB]` atoms).
2. **FoldRun High-Level JSON/YAML Spec (`pdl1_highlevel_spec.json`)**:
   ```json
   {
     "design_mode": "binder_design",
     "target_structure": "4Z18",
     "target_chains": ["A"],
     "binding_residues": ["A54", "A56", "A66"],
     "binder_length_range": "80..120"
   }
   ```

When you run `!rfd4-generate ...`, `foldrun-shim` (`foldrun_shim.cli:rfd4_proteina_main`):
- Accepts both Hydra overrides (`inputs=pdl1_binder_design.yaml`, `out_dir=./rfd4_out`, `num_replicates=4`, `seed=42`, `~metrics`, `compile_lengths=null`) and standard CLI flags (`--num_model_seeds=5`, `--design_mode=binder_design`).
- Strips local checkpoint overrides (`hf_ckpt_path=...`) so the container uses `/mnt/nfs/foldrun/rfd4_proteina/cache`.
- Inspects your YAML/JSON input via `foldrun_shim.bundler._collect_rfd4_proteina_assets`, packages any local `.cif`/`.pdb` target structure (`input: ./local_pdl1_target.cif`) into `assets.tar.gz`, and stages it to GCS.

In [ ]:
import json
import urllib.request
from pathlib import Path

# 1. Download or write local target structure (./local_pdl1_target.cif) to demonstrate automatic asset bundling
cif_path = Path("local_pdl1_target.cif")
if not cif_path.exists() or cif_path.stat().st_size < 10000:
    try:
        urllib.request.urlretrieve("https://files.rcsb.org/download/4Z18.cif", str(cif_path))
        print(f"Downloaded 4Z18.cif to {cif_path} ({cif_path.stat().st_size} bytes)")
    except Exception as e:
        print(f"Failed to download 4Z18.cif ({e}), using inline fallback")
        local_pdl1_cif = """data_4Z18_cropped
#
_entry.id 4Z18_cropped
loop_
_atom_site.group_PDB
_atom_site.id
_atom_site.type_symbol
_atom_site.label_atom_id
_atom_site.label_comp_id
_atom_site.label_asym_id
_atom_site.label_seq_id
_atom_site.Cartn_x
_atom_site.Cartn_y
_atom_site.Cartn_z
_atom_site.occupancy
_atom_site.B_iso_or_equiv
_atom_site.auth_seq_id
_atom_site.auth_asym_id
ATOM 1 N N PHE A 1 -14.120 18.230 -8.410 1.00 32.10 18 A
ATOM 2 C CA PHE A 1 -13.010 19.110 -8.120 1.00 31.80 18 A
ATOM 3 C C PHE A 1 -11.720 18.340 -7.910 1.00 31.50 18 A
ATOM 4 O O PHE A 1 -11.680 17.110 -8.020 1.00 32.00 18 A
ATOM 5 N N ILE A 37 -5.420 12.110 -2.310 1.00 24.50 54 A
ATOM 6 C CA ILE A 37 -4.210 12.890 -1.980 1.00 24.10 54 A
ATOM 7 C CB ILE A 37 -3.110 12.020 -1.340 1.00 24.80 54 A
ATOM 8 C C ILE A 37 -3.680 13.540 -3.240 1.00 24.30 54 A
ATOM 9 O O ILE A 37 -4.390 14.310 -3.890 1.00 24.90 54 A
ATOM 10 N N TYR A 39 -1.120 10.450 -0.820 1.00 22.40 56 A
ATOM 11 C CA TYR A 39 -0.050 11.210 -0.190 1.00 22.10 56 A
ATOM 12 C CB TYR A 39 1.240 10.390 -0.080 1.00 22.60 56 A
ATOM 13 C C TYR A 39 -0.480 11.720 1.180 1.00 22.30 56 A
ATOM 14 O O TYR A 39 0.310 12.310 1.910 1.00 22.80 56 A
#
"""
        cif_path.write_text(local_pdl1_cif, encoding="utf-8")
else:
    print(f"Using existing {cif_path} ({cif_path.stat().st_size} bytes)")

# 2. Write official native rfd4-generate YAML (pdl1_binder_design.yaml)
# Uses contig + C_CRDGRP, C_SEQ, and C_HOT conditions from nvidia/RFD4-Proteina
pdl1_binder_yaml = """binder_design:
  input: ./local_pdl1_target.cif
  contig: "80-120, /0, A1-115"
  conditions:
    C_CRDGRP:
      1:
        - select: "sel('A/*/*/[N,CA,C,O]')"
    C_SEQ:
      true:
        - select: "sel('A')"
    C_HOT:
      true:
        - select: "sel('A/*/37/[CA,CB]')"
        - select: "sel('A/*/39/[CA,CB]')"
"""
Path("pdl1_binder_design.yaml").write_text(pdl1_binder_yaml, encoding="utf-8")

# 3. Also write FoldRun's high-level JSON spec (pdl1_4z18_spec.json) referencing RCSB PDB ID 4Z18 directly
pdl1_highlevel_spec = {
    "design_mode": "binder_design",
    "target_structure": "4Z18",
    "target_chains": ["A"],
    "binding_residues": ["A54", "A56", "A66"],
    "binder_length_range": "80..120",
}
Path("pdl1_4z18_spec.json").write_text(json.dumps(pdl1_highlevel_spec, indent=2), encoding="utf-8")

# Inspect how foldrun-shim bundles ./local_pdl1_target.cif into assets.tar.gz
bundled_binder = bundle_local_inputs(
    model="rfd4_proteina",
    input_path="pdl1_binder_design.yaml",
    cli_args=[
        "--design_mode=binder_design",
        "--num_designs_per_seed=4",
        "--num_model_seeds=5",
        "--seed=42",
    ],
)
print("Wrote pdl1_binder_design.yaml and pdl1_4z18_spec.json")
print(f"Bundled local target assets     : {list(bundled_binder.assets_map.keys())}")
print(f"Deterministic client_input_sha256: {bundled_binder.client_input_sha256}")

In [ ]:
# Interface 1: Drop-In Native Hydra CLI (!rfd4-generate ...)
# Runs in fast mode by default (compute_metrics=false / ~metrics, skipping self-consistency refolding).
# Pass `--metrics` (or `compute_metrics=true`) when you want full self-consistency refolding metrics (sc_score).
!rfd4-generate inputs=pdl1_binder_design.yaml out_dir=./rfd4_cli_out num_replicates=4 seed=42 --num_model_seeds=5

### Interface 2: Python Notebook Helper API (`from foldrun_shim import rfd4_proteina`)

For interactive notebook workflows, `rfd4_proteina.predict(...)` submits or re-attaches to the Vertex AI Pipeline job, displays a live progress bar, renders an HTML summary card (`GatewayJobResult._repr_html_()`) with a direct link to the **3D FoldRun Viewer** (`job.viewer_url`), and extracts the ranked designs locally:
- `design_mode`: `"binder_design"` (default), `"motif_scaffolding"`, or `"unconditional_monomer"`
- `compute_metrics`: `False` by default (explicit fast mode `~metrics` / `compute_metrics=false`, skipping expensive self-consistency refolding and reporting uncomputed self-consistency metrics such as `sc_score` as `None` / `N/A`). Pass `compute_metrics=True` (or `--metrics` on the CLI) to compute full self-consistency refolding metrics.
- `num_designs_per_seed` (or `num_replicates`): `1..20` (default `2`) — passed as `num_replicates` to `rfd4-generate` inside each GPU task
- `num_model_seeds` (or `num_seeds`): `1..50` (default `5`) — number of parallel GPU tasks in `dsl.ParallelFor`
- `seed`: Base random seed (`int`)
- `gpu_type`: `"auto"` (`A100` for $\le 1500$ tokens, `A100_80GB` for $> 1500$ tokens), `"A100"`, `"A100_80GB"`, or `"L4"`

In [ ]:
# Interface 2: Python Notebook Helper API (5 parallel seeds x 4 designs/seed = 20 candidate binders)
# Default fast mode (compute_metrics=False / ~metrics) skips expensive self-consistency refolding;
# pass compute_metrics=True when you want full self-consistency refolding metrics (sc_score).
job = rfd4_proteina.predict(
    "pdl1_binder_design.yaml",
    out_dir="./rfd4_out",
    design_mode="binder_design",
    compute_metrics=False,
    num_designs_per_seed=4,
    num_model_seeds=5,
    seed=42,
    gpu_type="auto",
)

# Inspect .foldrun_job.json written for disconnect / interrupt recovery
state_file = Path("./rfd4_out/.foldrun_job.json")
if state_file.exists():
    print("Saved interrupt-recovery state (.foldrun_job.json):")
    print(state_file.read_text(encoding="utf-8"))

# Render interactive HTML status card with FoldRun 3D Viewer link
job

---
## 4. Mode 2 — Functional Motif Scaffolding (`motif_scaffolding`)

In **Motif Scaffolding (`motif_scaffolding`)**, RFD4-Proteina embeds a fixed functional binding epitope or catalytic triad from an input structure into a newly generated *de novo* scaffold:
- Using `scaffold_motif: "25-45, A1-20, 25-45"`, RFD4-Proteina samples a 25–45 residue N-terminal flanking region, preserves residues `A1-20` of the input motif structure, and samples a 25–45 residue C-terminal flanking region.
- `foldrun-shim` automatically bundles the local motif structure (`./pd1_epitope_motif.cif`) into `assets.tar.gz`.

In [ ]:
# Write a Mode 2 motif scaffolding specification (motif_scaffolding_spec.yaml)
motif_spec_yaml = """design_mode: motif_scaffolding
target_structure: ./local_pdl1_target.cif
target_chains:
  - A
scaffold_motif: "25-45, A1-20, 25-45"
binder_length_range: "70..110"
"""
Path("motif_scaffolding_spec.yaml").write_text(motif_spec_yaml, encoding="utf-8")

bundled_motif = bundle_local_inputs(
    model="rfd4_proteina",
    input_path="motif_scaffolding_spec.yaml",
    cli_args=[
        "--design_mode=motif_scaffolding",
        "--scaffold_motif=25-45, A1-20, 25-45",
    ],
)
print(f"Bundled motif assets: {list(bundled_motif.assets_map.keys())}")

# Submit Mode 2 Motif Scaffolding job across 5 parallel seeds
motif_job = rfd4_proteina.predict(
    "motif_scaffolding_spec.yaml",
    out_dir="./rfd4_motif_out",
    design_mode="motif_scaffolding",
    scaffold_motif="25-45, A1-20, 25-45",
    num_designs_per_seed=4,
    num_model_seeds=5,
    seed=42,
)
motif_job

---
## 5. Mode 3 — Unconditional Monomer Generation (`unconditional_monomer`) on Cost-Optimized L4 GPUs

In **Unconditional Monomer Generation (`unconditional_monomer`)**, RFD4-Proteina jointly generates *de novo* monomer backbones and sequences without a target structure (`monomer_design: {contig: "100-140"}`).

- Because unconditional monomers have a small token footprint ($100\text{–}140$ residues), you can pass `gpu_type="L4"` (`g2-standard-12` with `NVIDIA_L4` 24 GB VRAM) for cost-optimized parallel sampling, or keep `gpu_type="auto"` to use `NVIDIA_TESLA_A100`.

In [ ]:
# Write a Mode 3 unconditional monomer specification (unconditional_monomer.yaml)
monomer_spec_yaml = """design_mode: unconditional_monomer
binder_length_range: "100..140"
"""
Path("unconditional_monomer.yaml").write_text(monomer_spec_yaml, encoding="utf-8")

# Submit Mode 3 Unconditional Monomer generation using cost-optimized NVIDIA L4 GPUs
monomer_job = rfd4_proteina.predict(
    "unconditional_monomer.yaml",
    out_dir="./rfd4_monomer_out",
    design_mode="unconditional_monomer",
    binder_length_range="100..140",
    num_designs_per_seed=4,
    num_model_seeds=5,
    seed=42,
    gpu_type="L4",
)
monomer_job

---
## 6. Downstream Visualization & Ranking Analysis of Materialized Outputs

When the Vertex AI Pipeline completes, `foldrun-gateway` (`output_packager.py`) collects all per-seed structures and confidence JSONs across parallel GPU workers, sorts all candidates by `ranking_score` descending, and extracts them into `<out_dir>/<stem>/`:

```text
rfd4_out/
├── .foldrun_job.json
└── pdl1_binder_design/
    ├── ranked_designs/
    │   ├── metrics.csv
    │   ├── ranked_0.cif
    │   ├── ranked_1.cif
    │   └── ...
    └── predictions/
        ├── design_seed42_0.cif
        ├── confidence_design_seed42_0.json
        └── ...
```

In [ ]:
# 6.1 Inspect Ranked Designs (ranked_designs/metrics.csv) and Per-Seed Confidence JSONs
def _opt_float(val):
    if val is None:
        return None
    s = str(val).strip()
    if not s or s.lower() in ("none", "null", "nan", "n/a"):
        return None
    try:
        v = float(s)
        return None if math.isnan(v) else v
    except (TypeError, ValueError):
        return None


def _fmt_metric(val, fmt=".4f"):
    v = _opt_float(val)
    return f"{v:{fmt}}" if v is not None else "N/A"


stem_dir = Path("./rfd4_out/pdl1_binder_design")
if not stem_dir.exists() and Path("./rfd4_cli_out/pdl1_binder_design").exists():
    stem_dir = Path("./rfd4_cli_out/pdl1_binder_design")

metrics_csv_path = stem_dir / "ranked_designs" / "metrics.csv"
pred_dir = stem_dir / "predictions"

if metrics_csv_path.exists():
    with metrics_csv_path.open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    print(f"Loaded {len(rows)} ranked designs from {metrics_csv_path}:")
    print(
        "(Note: in default fast mode `compute_metrics=False`, uncomputed self-consistency metrics report as N/A)\n"
    )
    print(
        f"{'Rank':<8} {'Design ID':<22} {'Seed':>6} {'Ranking Score':>14} {'pLDDT':>8} {'ipTM':>8} {'sc_score':>10} {'Clash':>8}"
    )
    print("-" * 92)
    for idx, r in enumerate(rows[:10]):
        print(
            f"ranked_{idx:<2} "
            f"{r.get('design_id', '')!s:<22} "
            f"{r.get('seed_value', '')!s:>6} "
            f"{_fmt_metric(r.get('ranking_score'), '.4f'):>14} "
            f"{_fmt_metric(r.get('plddt'), '.2f'):>8} "
            f"{_fmt_metric(r.get('iptm'), '.4f'):>8} "
            f"{_fmt_metric(r.get('sc_score'), '.4f'):>10} "
            f"{_fmt_metric(r.get('clash_score'), '.4f'):>8}"
        )
elif pred_dir.exists():
    conf_files = sorted(pred_dir.glob("confidence_design_seed*.json"))
    print(f"Found {len(conf_files)} per-seed confidence JSON files in {pred_dir}")
    for cf in conf_files[:5]:
        data = json.loads(cf.read_text(encoding="utf-8"))
        print(
            cf.name,
            "-> ranking_score:",
            _fmt_metric(data.get("ranking_score")),
            "plddt:",
            _fmt_metric(data.get("plddt"), ".2f"),
            "iptm:",
            _fmt_metric(data.get("iptm")),
            "sc_score:",
            _fmt_metric(data.get("sc_score")),
        )
else:
    print(
        f"Output directory {stem_dir} not materialized yet (run against a live FoldRun Gateway to populate)."
    )

In [ ]:
# 6.2 Plot ranking_score, plddt, iptm, and sc_score across RFD4-Proteina designs with matplotlib
import matplotlib.pyplot as plt

records = []
if metrics_csv_path.exists():
    with metrics_csv_path.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            records.append(
                {
                    "design_id": row.get("design_id", ""),
                    "ranking_score": _opt_float(row.get("ranking_score")) or 0.0,
                    "plddt": _opt_float(row.get("plddt")),
                    "iptm": _opt_float(row.get("iptm")),
                    "sc_score": _opt_float(row.get("sc_score")),
                }
            )
elif pred_dir.exists():
    for cf in sorted(pred_dir.glob("confidence_design_seed*.json")):
        d = json.loads(cf.read_text(encoding="utf-8"))
        records.append(
            {
                "design_id": d.get("design_id", cf.stem),
                "ranking_score": _opt_float(d.get("ranking_score")) or 0.0,
                "plddt": _opt_float(d.get("plddt")),
                "iptm": _opt_float(d.get("iptm")),
                "sc_score": _opt_float(d.get("sc_score")),
            }
        )

if records:
    records.sort(key=lambda r: r["ranking_score"], reverse=True)
    labels = [f"#{i}" for i in range(len(records))]
    ranking_scores = [r["ranking_score"] for r in records]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].bar(labels[:12], ranking_scores[:12], color="#1a73e8")
    axes[0].set_title("Top Ranked RFD4-Proteina Candidates")
    axes[0].set_xlabel("Rank")
    axes[0].set_ylabel("Ranking Score")
    axes[0].set_ylim(0, 1.05)

    paired_plddt_iptm = [
        (r["plddt"], r["iptm"], r["ranking_score"])
        for r in records
        if r["plddt"] is not None and r["iptm"] is not None
    ]
    if paired_plddt_iptm:
        p_vals, i_vals, rk_vals = zip(*paired_plddt_iptm)
        sc1 = axes[1].scatter(p_vals, i_vals, c=rk_vals, cmap="viridis", s=70, edgecolors="black")
        plt.colorbar(sc1, ax=axes[1], label="Ranking Score")
    else:
        axes[1].text(
            0.5,
            0.5,
            "ipTM / pLDDT not computed\n(fast mode `compute_metrics=False`)",
            ha="center",
            va="center",
            transform=axes[1].transAxes,
        )
    axes[1].set_title("Interface ipTM vs. Monomer pLDDT")
    axes[1].set_xlabel("Mean pLDDT")
    axes[1].set_ylabel("Interface pTM (ipTM)")

    paired_sc = [(r["sc_score"], r["ranking_score"]) for r in records if r["sc_score"] is not None]
    if paired_sc:
        s_vals, srk_vals = zip(*paired_sc)
        axes[2].scatter(s_vals, srk_vals, color="#34a853", s=70, edgecolors="black")
    else:
        axes[2].text(
            0.5,
            0.5,
            "sc_score = N/A in fast mode\n(pass `compute_metrics=True` or `--metrics`)",
            ha="center",
            va="center",
            transform=axes[2].transAxes,
        )
    axes[2].set_title("Self-Consistency (sc_score) vs. Ranking Score")
    axes[2].set_xlabel("Self-Consistency Score (sc_score)")
    axes[2].set_ylabel("Ranking Score")

    plt.tight_layout()
    plt.show()
else:
    print("No materialized metrics to plot yet; run the prediction cells above first.")

In [ ]:
# 6.3 Interactive 3D Visualization of Top Ranked Binder + Target Complex (ranked_0.cif) with py3Dmol
import py3Dmol

ranked_0_cif = Path("./rfd4_out/pdl1_binder_design/ranked_designs/ranked_0.cif")
if not ranked_0_cif.exists() and pred_dir.exists():
    cif_candidates = sorted(pred_dir.glob("design_seed*.cif"))
    if cif_candidates:
        ranked_0_cif = cif_candidates[0]

if ranked_0_cif.exists():
    raw_cif_text = ranked_0_cif.read_text(encoding="utf-8")
    sanitized_lines = []
    for line in raw_cif_text.splitlines():
        if line.startswith(("ATOM ", "HETATM ")):
            tokens = line.split()
            changed = False
            for idx_tok, tok in enumerate(tokens):
                if tok.lower() in ("nan", "inf", "-inf", "+inf"):
                    tokens[idx_tok] = "0.00"
                    changed = True
            if changed:
                line = " ".join(tokens)
        sanitized_lines.append(line)
    cif_text = "\n".join(sanitized_lines) + "\n"

    view = py3Dmol.view(width=840, height=540)
    view.addModel(cif_text, "cif")

    # Style Target Chain A (PD-L1) in slate gray cartoon + translucent surface
    view.setStyle({"chain": "A"}, {"cartoon": {"color": "#9aa0a6"}})
    view.addSurface(py3Dmol.VDW, {"opacity": 0.35, "color": "#dadce0"}, {"chain": "A"})

    # Highlight Target C_HOT Hotspot Residues (1-indexed label_seq_id 37, 39 / auth_seq_id 54, 56)
    view.addStyle(
        {"chain": "A", "resi": [37, 39, 49, 54, 56, 66]},
        {
            "stick": {"colorscheme": "orangeCarbon", "radius": 0.25},
            "sphere": {"color": "#fbbc04", "radius": 0.6},
        },
    )

    # Style De Novo Designed RFD4-Proteina Binder (Chain B) in vibrant blue
    view.setStyle({"chain": "B"}, {"cartoon": {"color": "#1a73e8"}})
    view.addSurface(py3Dmol.SAS, {"opacity": 0.45, "color": "#1a73e8"}, {"chain": "B"})

    view.zoomTo()
    view.show()
else:
    print(f"Structure file {ranked_0_cif} not materialized yet.")

if "job" in globals() and getattr(job, "viewer_url", None):
    print(f"Full interactive FoldRun 3D Viewer URL: {job.viewer_url}")